# ACE-Step: uji konteks dan penyambungan bass

Gunakan **VM Colab CUDA yang sudah punya checkout bridge/runtime/bobot dari notebook 10**. Jika perlu, salin sel-sel ini ke notebook pada VM yang masih aktif. Tool ZIP ini bukan installer model. Helper bridge sudah disertakan di notebook ini. **Upload piano asli saja** saat diminta; tidak perlu mengunduh atau mengunggah ZIP tools.

Semua flag inferensi awalnya **False**. Konteks **90 detik sudah dipilih pendengar**; 120 detik dilaporkan banyak noise. Jalankan sel persiapan lalu bagian chunk seluruh lagu. Probe durasi bersifat opsional dan boleh dilewati. Kualitas sambungan lagu penuh belum divalidasi.


In [ ]:
from pathlib import Path
import base64, gzip, hashlib, importlib.util, json, sys, tempfile, uuid, zipfile
from google.colab import files

for dependency in ("numpy", "scipy", "soundfile"):
    assert importlib.util.find_spec(dependency), f"Kernel notebook memerlukan {dependency}; jangan ubah environment model pinned"

# Generated from repository scripts by scripts/build_context_bridge.py.
# BEGIN EMBEDDED CONTEXT HELPERS
EMBEDDED_HELPERS = {
  "ace_context_chunks.py": {
    "sha256": "1819d968044da24cf159720300ad9bd11f7d008ebd55036378f23447c1cc73cf",
    "gzip_base64": "H4sIAAAAAAAC/6U7247cxnLv+xWUzgPJ3R7ORRfbs0sBPorOiZHEFmwnCDAZEE2yZ4cRh6R52YsVAfmH/GG+JHXpJpsc7sqB9bAaNrurq6vrXsWXL19+//7D4pdWVU5cdkWq0kVSFq16aB31UKk6O6mibRxZpM5JyaarVSqcRp6qXC1knt0WKnWaNmuTY1bcBhcXP6uuUY2jcpW0dZZEhy7Po6aEd9Wj0xXJURa3Kg2cj7ksllVdxspJ1UF2eeu0pVOUTlYcVK2KRAUXPyqVNk7RnapH2DPJ6D9E8pDlSjiHw6lSt8Kp8q5x2qMChDNApbh13v/rP3zv1F3RAvbOoaydxUI9qKRrAejLly8vslNV1q0j69tK1o0yz0fZHPMsNo//CWib3yfZHi8OdXlyKvgFkxz94iO+0L+bY9dmef/UxXC+RDXNhRmiozgSzlT1s8x5cLg58B501qAB6srcbJSUda1y2eLBsyKNKiU/DZDP6U3wgBi5gkv5/tcPTui82qxWq4uff/rpV3hAxL0owq2jyA9q1ZT5nfL8AEiCV75b7y8uLuBu4Fhy8+athwf3txcO/EuzW9W0oaZXoCf49O4+a48MnBYEZaUKz61j1yeM2lrJE0PBf3g5cV4mn+DenaxVtZfLU5zKLU8EtGTqrVeb15f4xxex6/pb3j7oqhTI4dFy3rtWbVcXGr3gqB74F2DGJ7mvYYsI75WQE3cy75Q+Ew6EA97XrULKyvoxxOcAjxU13eGQPXhu0J4ql7fspwUMHCXHwx2CFC678WgLARcGNA03QuZ5eR8Vsgj/JvNG+VfufxRnkGpV5TJRjIhGPZePZdd6bdnKXDhaRMPvVgJYs77NivAt8IVMVRi8+Rbed6oJfywLYBZVqPr2kR70SQmlJvQ0EMEABK5mTLKD48ni0cuAf5pWgix6dyIuy9x34LqKsnVGb7ysaMUhL2Xr9xNQXoKsAU4Fonh3Pl30HV4y7+73HKAXrFc3oUboJny7Wpnx1Q3idcM44iD/ulnjb71gsbk0oyt/4K1aZo1y/g33+1DXZe25qFDMItjxf//7f2CnxpDwXbhumIiIp7fShPGR3LWCt6vG7Qk0oQLfCxDCHw50iWJ3E9IrOhMNPIffL2VXJ8o5dU3rgGI0GDqNAqRTs/tfnB/gBLcKaFGmKnfSrpZtVhYNoJ3kHaCvQFtXMk1RJTs/qjuYqh4SPDyqyVr9BjfQIilkFRDIk3zITt0ppGuDqwRkNJl8QlpLSBrWqK48Jgy98a+JOqGGAFcB0wyR8M3N+uvnNucxVwN4ObmSd2BIctCfgLUsgBqNUzJMR3ZpVmpyJIBSG65xP01qjQsoRdgHHryNoIMlKss9virG018iNF/DkZVMsjYD0djh8BVMQKAZcsNuJWifxXrPYAkNZGp6XaNV82iGvx9Oj4iF660NWePGk8AiwdBqf+1UZZ4lNMG8od8EBn4CqzkowXpvGDGbTOS0fVJO2z8gp+0gloNkrm7aG6LZkswInrnFMyMWBAd/vAgbMEMq9RrVevTmWUH8K7IRqDoHLXTTMzyjIcBPyIBF0dgjTHI+4BwZcDbyL1o3ffeTC1jzJdlbq5MEoMUti+ECKb6DS+wntMDKqg3N+BUzeL/scri7XQbvl0138sZj270/kC0vkfu8HpxmfsG7TxePlh6z8JQVw9LJzhoEM9ki8y816AEAcENevjtm2xnFVzoH8N6yGPyM2NAevb16qhFYApda3+bZKWuN3rHIhYgeM4FHzUvBgyMqCDyN/RYEjrAVeimPXm0mZygr0mThju+h5ddjpuOD3oSjKTfhMdvbpNCQtjZHo8AR7vql+KQeQ3Y4nHYr48ZrF+YwWiYfQxe8uDvgvTQylHN7mCiOkx2AWTP0S5owB2/UY8YkglytBaK6XIKVIsHZzZNxP4vyAHiMtccGHmQXT80PrCZW/lWwWm0uR+dajulNel2fEz0TXh8VStYRr3CnYN1Y5qhR0giFd6AFMm4gK3D3UmRif1BqZpQ38i+ms4mz9fCxKz71WpDEW3hSxD5evgL/WdXo8/2eVSQogqSFRHC4BlB4dSsAMHHgSsgFqnpiPDbTMar2gQbGfoZAynRBq5lM/QzZNEoHAWxEzBK/d1oGMtAJzNE+u+j6PbjbTLhoMiIC726lfoQ57jYWMKvqWvOSD6DHaAb8EaM7s/655AVEBiV3O4Pml5F//NnlAC5CWrpbUlAukSY61BIUsrtlQrn6cBHoosg4IVvjN16MsSCNMczSXqWL/tQwik/CNYIUGQ5xt+YX7EkEhG3o/y/a/8VAICKzPwpEZCtFU4fNgSMF8uplfi8fm2iThr/WYEXS9rFSoUuG79Vm8OCa+kVIFg2YDOFgGFMpCHtehBtj+4pqMI44xw/Agfe+7mPyEuBECGVLiLs+/ePvI5/FBCoAUp+vAqcfAi+vIRdQOODsw+3/MU+fgjjbw9dAGAYHNTxkxXmChnmKNUzreTQ4fUqz2tPxIBOTQuyo/KTDF55dZ4AWMA/Gg6EOBvV+NOEW1QcxYkQ8HTL8pccs7l7x5IDDqwD0EMSB2pxwTB0kZfWIpzT0mYIcbpW3P3v/Ihyh+dwN/qQnOrTS0SkL5/6IUTrfEyYZ0HBCnGbCtwQc1sKczNUnupd3rvbZ38uiLLIEwBIjoEeH6g0hoKOxPGUPCLUs8sfA+TujDx4RpgNoJ7ghVd+pVIMzp4G7Jpw404AQQPyTo+KMCJogBLUAdQRWDHM1lBIAjQhuNaVfGo4AhmxFUHeFt3M5u+IKd1GUTQuhBP68gz8KqUTj+CdzBYTq5rphQNY8wtZ+VmnBpATWbnB9spXwX5Wcomb9Nle8lmjp7wWcI/lEjMc0JsqFlirgiYPHjDLQu99GMoa7PqlWosyFFKCDQkgbj8Sgn+uTHuEo3h+5VmZtADbR0/cbMbe5wF1jth/7BM/4vwaqk5ZwZdojT450edYVU1iIfGu5YnTgMVZpeV/ESoLHJnZ7/2qCskrI6aF3YyhnvjsrV2UMv0dmwToV5VxCuoMdG1masWD/5lt/i8Y2V4VHU3x+e2Xe7i33nPQg6WYPlG3zW03/n+BaObFzebnxNT4oKqHOg1iwZ9IYAk8hGPdhrUkYNcBWJxlBVNyg1d8Ay8m2a8DRQ1lUqSt6VcG3OlYdQt89KV1b0M+ZfcQlhkWYZwUJdBuyPAYff/7wy4dfzyFYeszedqreggLoK84nz+J/tgmwHZzkR517rRWmesFyLXt2Cpz3VmSuswyNtkeNjs84hB+lHn4A3wfdILAf2i+6JoOWFV3WPnJSuWtIKf7WyRzHugLuJTtkKg00r1tJO6Nd6TZxxBX4c2RTcUDb1DN6eIcyh5iHF1lpPz2+JLgkLPN05+12rj0ErufEemLePAF5Cp8DpzXHGKAe3I+yTBWH6BFlf7M7FbWlxnfYl8IJO0/7IjRYPGfp/t6jZXSMMXYATjVw2dqOgEkYey64iclMghqN8Bi2YzbJpg6YXpMUW/qXjnemdBkdTr2yA7KcIb1NJi2IQx57QjILlE0trbBfhLMX8RzxBoP+R2kHmPIuLPsueJoj8Se0eID98Cl2Ix/9WeQ+0g4EkJEzoYCTZgcQ8oY9C8t/ORcWi2IzYjZ5qbWi5gmS4ceIZN4WOkEjmkfAawG7Y/OCkUOXqRChl8JyPmOXwbM5hWZBX/+IcQlq4+ve2PM+LIjscIrPX4ZMt/2aLQHacyOyYA6QivYkfXv9rZjbtFPaY9M7sSdm6Z/QDbPbcPgIwInKu1FQuR9HtGdr+xTQEEgCIG8ECUPR/WIOuP8V6PoMtzIjsME351MmUexwismLEZkn0gIcMT390ziP4GhlwADGHKFO/bU9G/i9xz0ccrMLzI8sgSmwGtksDe5LNL/OKWvIxxsrVNzHhLpdwZLTeHVZsnhFlm615UnrSUsH95NtPfq8JWRR4ELiISnafCjk9RIZDCAC9MtcU9mTV1zZw0GrrodgsJKQfPLwj+CBf/7p/T9FH/79v6ynH/86cDQQ7xaESQe0ZmszyprgbPLImNgARiqDlO/oLYWyjedzVuvzl1FNkuiPOSeWQZ2T2I/9eszDUbhiuAszPXswcaQWDf4HXh195mkvadrL/Xb1Kv3ijhNxtE6H3X2kPYQ/VjQCWxN2+kTbM58ukeDlpoNnoyfuYOF+6rBMYKM15eXnNpTM53bWGuICMgu99aMDu+d7jKwD72Rbh7P5FcTbIIrUWuAY91BDx2SoOORdc2RCXWv/Uo2g/MX5WSUl1sEk5WYaxFW2WHJtHXlo8UVBPmxddxUmwg3B2P0MxinZErNn6OH18abNOwZuVugChqZbcJuXMQgMv44uR/pkiIjIKOpJM5Zw7sa0wsqYs4Gl0bO2xEK/t+VhbO9C295tZ4P20aVp9Oxbux6oot9ex7Dhpynn9tNMTet8O63VWfFXj+0RXH/yJVEdgijDdR5kljuxOmAVLoFtqO0DrhD3fTQ3cB7odKdY1eHq7AXndvQ9gcSaO/rMK1hW/V5lbHn4Klyfy11PheeBPXVocw2s+nW2qwcqyHY9bd/n8y1Dgvv/Zc6F7pmZ0UDnUtxj+JQgjzQQq02zJjgPbXz/2m7WsBW3ME9jf1QPahuaxp7d3DHKM2xWl0UFknG7XnmYvOBRni7WarGmjIO2xadG5xlmAE0TFvTcYRaXlujMM4zS9Lev/QFuhR1TI8insig5o8LAJPBauPZ32+23+0A2CMqzQBn3FRMhuNS/efuaHFVAmJ/hJK+3fcb/+Lu7pYYQF5u6yjRLINp2t6vgS7/7IsS/vLvOtyVh33ZEUIX+k2JCvctzV4ATdyxTeDqA57nrscGyqAaxRArosre3FsMUGJa6PGbGtGufl/fimN0edYnv9Wq1Wm7erLiGY429euMTPJnAhjrRg21RIgqHHil4uwOAWwSIxhnHrsBjurejR4RCbyg2hBWr/U24+sPki7EdipkC1hKgPZ4ZFO11Lm9DtF+Ie8U5XzLdOIkq+LBg/y5EEJfBd29MN0JZHLIU++DCntmSPKvoNPJ2vyQcxUqs/UltB1El6sA0VrgG0rswePOG3Z2ZswzzTM2FSjf3CugGSs+urEtUVXiteVY0FbYqrQQ8VdlywxX4c9a3ccSjlHAvCAUYXCAyewQAOnAyalrQlDxF93Cl5T0wy6EVNSIlctIzoub/EoX2W9xnae8qa3VJLxb0YrncXGNpkN5c0RAfKUS4Oxpe5HNKEYtw82+Y0+OQkNIg6idB1E+DQG7gMhSEXoSb2BBDxmejM0X+v2HnElh32KQBjnwswQmIS/DHdIbU1PHvpbiPw9HlMs2u4xxJIy/v5VV8eR9zNksKBUdDPegL/C/2r2t14FYhUn+ekpdKXqn4UsX+cmP4t64HzmWNNqPGLmEFrY/ZRUchoR39d6C/vhmYlfkHzVgIup3wQGT9JeDisw90sJasAl0n+Qjj4N391mWKqmlam6VOC6qnybgNAIQRovPsd6XpheFbnnO+BF2puMvytNO9UsC6bejRgbHAzvV1wsy/8vDZWy9wH193QNXo6KCOGRp4VpPQT9erBR1JfHaZYa2KKz2zUXaRTyMgQJTGB3gH1FDSKrS4xIWTCbFP9Wa2s1gjxgfh0n6Dh89r3C0dxoJY1QqUBOYGcnUHLhknumgqAV+ipoNb07YTt2pad8u3j799o1G4UziSdS0fG4pVBZm8RsurLv+Pg65r4k5tC/jVqKi8t3pItEXkpRDK93WCZk5iPoAP86hjPdODRNmconXtSigqst9VDTrLm0vEgThOFN6rzdCfxMpJF/0Kh1oXCL3x0Y38kwdgpP0P+mtaS0wr1uxeTEvWz2UvGEvafVmgPaAy9iRjQepSxMYLtdoa9sIeI3yHLAtRcie38V4XjuTsWbbx4mkFiXZg3B1iN39ZN2+dd7AXjLFupzK/99dFecLknFbfo/MMPSVKgt427Y+gpNbBG52fmzQ+7Jev/ElTk9UT5O30bvshG3zeDbEnFMNnW4/sf6RUmAoa/IIRFvrxih/7DqTNyipuYnDcSqCjRD8IdLRzlPmBctemIdO033Zt34VHfbKwQkH87qj01oqR2zo7jcR1SiKij7EUEwrtEpYa6vbqh0mX9hb8nbmrEYtc4b6z9EEdnlyxleuB+Dds8kfShSBGbWScDDFozLfWIVdyuow6GOAKi5ZI4qRZk8g6tQSHPEVkIM/2aVgTMGOa33theTqJdmvmaOPbDWEP24fdq/2OVfB+JLDavGDM35C7aXHAD2B6sT9AaAdC3z5YjESX7twG7rWjJkHKk2TYklfSvELdEzsEU2EfM+NWasnng07eLuzLMLpAzo2e6ZR4O2FzotWw2X4XL+YYZnbZ7Mx5NUb0tG4WsxGjE5r+s5lD6FL5K/9rZ6Ts1NePMj+sS+6vBnB5Jeoq5PCTMfaFDkZxI0skqelnvVmtLsnVg3h549XVDuOL/TLXP8jJMQ8kZ2bKxHXT3njbmCo8pUHDzGgooxxC/czODtt5PAsb21DyELotPBA/0RdHXhIdLMQjk0/Ej3XFBzZuDB+U/j4B6ySLDjCs1V2m7kOPjrF7wnnaOzfO4hUFkXFjpj7tQO39d2+fauwb/UOxpy+xbGcS6Y3b0BvwmDFEnofGJf6PeO4l4aCrlVjEf0S3R9fhA+cDVuIXFfZgDU0ptx1G6q0C9+goG2UV8QNLt5F1Np2PdPQhJrdiARAKDP0Bb919pm0RVodCtq0CAuElLuO8PfxgdloHtuPMyy9xHTjNtLu7pf+Ei7lWbHhQMq3L8sSlpy1NxeugZ8oPuFv8+2R3peMi6dyt+wt/jZVm8hYborKkoV6ta/xmTXZtCUKSJU6Sy+xEnykcyxouJ0m6WiaPAhvd45oI11BPVvJIJbBx90PgThxlbMX6M+WfuRqJN19XmamwstKx3a2vlkUA4fOaw0yNZLYGoc0twJjvd3i+lDLn4OuanF2UGMrxwxENz1qdZaPsJqyx85r+KDboa9Z0ZSp1TSCiuyj7pPHM9z56iZ7KHzA2132zn5O1JsktgTJHEEnexbUuSOiSwdOx1XlPpy4OHvijtb6v5ryKziDIaxRNF3Mr7cf3/xJtXrtPADllD/QVIq8P3lx6wTeXFmV1T8OVbt96BjRyHNZF6QOcMXKu6Pex2e9PdjHqIxzcz7jtFzqB3zcp5ll8ql6Bn6VwSUxj6+82n+bXwlR3poWRzoT+IsYtdHE7rbmsY3C8r11ofLubpgT2kxBjegsHAhp9psUjObGuVfsnOj32ikFOW/f47ZV+u3/uvnSZybTZ6a66Xi6EesD3OmwOrT2sIrvpknuaJf0n28LY/nBfGOMyyrPwkFaw+H2PiYqr0HwJHHxf33b4xfNHfKq9FLRGndHHImEUpWUSRf41HD4E+5WmEbIbTWxwZhuC130CZyHF7X/rMrD61s1T4xOsoJW8zHO5AYeaomhc6v09d7FgQXEFURrrXxOo86tK7G95asnQCzldZj6lNAzGIMhoC/2Bdvjd6ok9Ob57du3bZ/ZGl+7ZxcGbb5/YGRs9reMOMoYNkaw3qMsZRBUVv9DcONIZ00vBpXDLZ3vRXT1PWmPFAMILw/ngkpyBwuKeDcocFL/NnttaV+JAbqmHF0FjlgICMuWe7cx5AzjzzMZ9Gw1vjl8EgqeSJRh4v12J71YCPP79lNHO4XB8MXdXOrcJkxuQEiIprmy8PkGHT4GWlJC7bl37i70GwZjvIWiyLoPSb+BvoSHwtymacXhwnGngMTtZqJf2XyCrfAYhJp7VD/KMw0Vrhy6aoY1jaP8geNFnmmnozy7L9R/p+OBAaahZP1mrpolnpWoatSvVmsDTejOhR0Xn55uFxLMb6wI13xrltfvi8+j8fW2ZRvXD01eCsjtOk1rLtvpEVt/UcJiZ+6GP9iLBq0T09GXSREOuwSGfzOLuFOuTfw3YfPNPNV9AOYpQOKMIjhNFaH6iCL+SQjN08X/qghc6h0MAAA=="
  },
  "electric_full_song.py": {
    "sha256": "961209740d96e8b853cbc0e3c34dabec30f7f08136724fa23f4055f6c9b2b605",
    "gzip_base64": "H4sIAAAAAAAC/70863Lbxnr/9RRbnB8BFQqk7DpnyoRnRpGlWD2xpLHsTKeqBgMCS2ojEEBxoUy7muk79A37JP0uu8DiQkpx03oyMbHYy7ff/QY7jnOjklUsj7KgKMTx9L//879+mE4LcRbLsMxVKH6G8bFQSRhXEcwUVSFFUIhAxGkiRZXFaRDJSJymcbAQ9zLOZO4dHHwqZCHKeynkZ1WUuC5TSQLz8iop1Rq2SCIhk5WCPU6uL8biUZX3aVUKVRYiUsEqSWFZKE5/vfiuODieTl4DTGGQeeIKVhRplYdShGlSys/lWCAk6zSSMUyJ4x9FksIUODO8r5IHPDzNBWxXhvfw4B04jnOg1lmalyLIV1mQF9I8h2m2Nb9/hz3M73VQ3pvfaXGwzNO1yGAsVguhh6+tKUW1yPI0lEVRj2zrn3h/3qHcZoAmPXwDf8fyMljLIgvCGqKqUpH5/Rhs6vEvKluqWB4cfDj5eCbm4vWr6XR6cP3h7ObsIzx+PRDwxynCe7kO/I3MC5Umzkwcj4WjIvjhSE3howVQ2IFhwiC+gdOLUmZHm+M3+FI6Y94skhsVSpwRVlGAS7Jchkrv7CyBFcrXr3BcJUWZV2uZlPjCHFBIGRUwcDu90zuu4HJBEkq/AMLhzn/1prR8KXNJ4wAHLnmDw+kSj4j8MvXDrILRj3kl9U7AG6WG46IAXiyB15ZAbbh4uY2BDw0/IzACzi2DXARhmK6zIFEIqSdORJRvxyIJyioPYpHFVfgA25TIXg4dY/1xkGNFGMsgj7fAR8CtFR8blGUQPoDQFPdIqCQtpcjh/ABlAvk+AElaxCg8ICziPsjXaaLCwhs45EQUxBZiXRUKcETgxyg1KgEZTNJ8rQeP9J1yuQKJg32XaRynj8j9KIeZCpL0u2LgCD4f7g18ibMRwmibBGsECUj/dHBwEMklSGURpsBIfp6mpYt8MRZAf6BULqP5JSBpNGNaOM65wmuKRa6iFcjpvQwfULiNkCNKFmn6IFZxugjiAuUTOKkCJt1IwCZwQ5IgLEaoYQeAxWHg8WjgcJQ3AmPk5RJIvpHuiN6vg/wByA5TXAdUxFKtigkgSJYFcbOHco3sWKZpXEwAV2tfboJ4AqrJy7bEqGGushLeybLKfK2y8B0fgNiABSpyUQXoW+OfHObnQJc4dumVmAAzrQFAVfgorO4IqJLTGNJPA8qbqmWDTaEKxJFApDa704763vXUkSc/AwdHoJZzt4sJvS9uNQgvwRwoQOcHvuNZnqe5u3Q+XF19FNsAKBCpTMXqXiyqhyBpKMmUnYmvuOOTM+qiAIdpDCgZgYyXEukB+NSQIRqAQSPgU0BEpMLSQ434ILeFq3mLaAXKGURzZMGMKwkTsM7lLcZmqwmQjkyDYw/dAIWLX2Qi8wAt0dF7ZKpCK7Xen0Ne6CFrug6akXWQ+ajw/MOJ3n3UwaFq84N1aS+IeLShDCMgl0G0BZTc0l3sSzWL2wDCIYapnI1MojSfnJyeHd2gpj723kw8GNxMFiqZZFsQs8Sx2K69E0r43q2IzFmqEpCBrjGYkEx6RbCUpUyKNC/sg+6Y6PcpGAqiOGhAGbl8W7hkc7ma62OZuHrBSPxNHFvi1GdN5yQCtSIXQMss6LKjJy4KJZB3/XcXlx9BTpMVMG0RxME9/L+sDI7VrK8H7T/fC9QC3u+AAReo74IxG5tLjWoCEqPrUTBpeBdzceAvSdKrdSfhzGe6uKg/NQPxCOAJx/YS1oG3rnOjFROvA9GXDh6bFh7plPlcOGBy+XjHZoUDSxXoxTXN9iNcP2lVvJBxtRYBECHD//32HrHpiX8GFCeoIAqYQ1qzdvTAvotFuoBz96LciVRtForJ8dQP0aH0jdn2l1Uc+yiNnsq2yWIs4iCuBFhc8CHFioS7UJ7Tog1fVJOArPK9jXxwyTxkiNsZEA8kEajsGjp0bIMzAmtuvdYGwgF+592XOfpttn4l9wAdNi/NgMNxNb0FxsoXzoh8aPAD0pmtqV0a8laypA1BYQF9xqIeTcJ7sIxA3tZoEayzRxWBWRiNxD+A2UOXcCxewX+Dyv43QJ0h79nnDDAMJgd9BpmCG/nw7ou4Pn1//ENfqzdw6PuO9PVBrNGFAR34RfoQGKxc1pTgVICHLUvEHrhIc/DiogrVcJrMj+FhFahkPvX+OgZb+NmvX0EE0vgSAGGsQlViwLE60j4/xBXsHRcAepqAz7DYgp8DdvMxRaP6I6iVVRBuRQxODYQUcPwWWBVYoKz9CHLBgQmQyERv44nn8t8rlUsapJkbRBg5FHSN5hJ8gzb0tay5QbJ1wTsGXxh9XHcDNg12HKEeRDlsv3IBOrBa6EOP6ikYdoCggiOrgBM27D5s0EowSG3FTlEOwCd+ElOzwxQeDGTip3kLUnwGVNtT8To4fNzTCjbTQHQnGCY+8G82YX+a22f8hMQki0Nbo9Gejo/vNG81cZIH+sK9tTHvOsvlOpMrlD3nCONBiD9RLx9t8P8SQaFHHlQOiygz3mjYujtHFInQPAScti71SE1AHAyW6EM4mxT0nZx/ReCfnF2bBrneAgWPl4cIFMVCR+GMY6Vw7RfHP8RSz2XJGN2N2YzNMZhhrACTVuDVz41W0TNB8eD+hr2CReHqmUc18smC6ol7KLh03sowxbi95oW1KoDbwvsZsT/a/Eh8NW+fgNGBQ77yeU9tNcuDWhOAY4o+Wy3/YJ2zalD+rYiBw+kcoosExcHIYXEfvHrzw1g85sBpPoXjf0xuAUuWkGnZ3SWD+vUL5JBZpyVuz0lL0JWXxiTXFLCcfnbRLOAahbML/GZG6wb7bG6tNfpXbPhJvz6eolroaI5nlUSL2KS8MBZ9vE9jVCAx8kfe4MZkn8CDADUfFRpHaa5WKgli/z4oMABitjCCzmhMlim8QRbxMEVQuJZmIfHymRH7KgZmLSTrmI5egfD90YfoI1dyZ7TgLDECL+fmorQwRd3h6DDT0kkg6mi7WNK1qG8CBX5OjAEtUczFq9zqbZ27W6fe+W4Eck7sYztzbbLV2xHdms1/QvrBiGszW81ozby/IVW96Zu9hL0hp6xHLZEu2UH4kYiMDhvRGEAE6wwnAbCgxBGStUoqCAFeIAMNaN8LBMyyZntB5AQh7EY5GIl8B34owtVoN/k5lHlWGi4j/vDWD5FCTw3DzoIoNeYcpp8+zM8DcKx5OiZ+0HHnZegTGjb1wOljByNM12u8BPiWf6Jh6xuxu31onNVgfA9wDFi7uxasPImsnzF+4J8RIN9k5QhP+oyOtddH9u0fxdKN+eMtbKYHFjokE4d6iGdrV+bwOctXGz5iVUyrlcicO1TPXtYE+8tns7ntWGHk1pfAobmwx6s1n2pQOIQ0sgauKJ/9vTn7GPwDy0GA8IzS9fPOwkMzn5YzWYixfJ1eahiaxxt2HohomDRWSGOEYNyZaJ2B0x/1dIRSRnaKC589iCBBAoN14bqvKI7hgAZzwJdXl2fIZ0gKTN7mwE2Io8bQPd6j0lnEafggZvMaIg9zEJqjiIFn5jRyL/hNHjy6tLKVP9O43LnA+bepA4h1Dc4PxT9qeBDnCjmCSfFV5+N/a6XjNaXh6Wuj7WdtumFSEmUdxunvpx2u6EIGJaXYwdY4UfqYtAZCYLBIPw2vfwxyhJXmOCcoIkcUXdkXofqKSsBZQy3KWePJg9xOaHuxlmUQBWXgOXdPzDe1++Y2zGVvaDKy9tho91LtYkZm2dfhKsdOVNY3b4AoCPknWD2wuP5pbE3x2e2oJ7a8EGbsp6Gt4S7NUr1oHxpGT1reNRqJbWoXqN6p5RQhnCiJvYO0fPZorW/Zm98RU0wsxNLHNATMYSF0SHH4zP4wSo8DJ1jot7WkqTP5ezhdJpGvZak5Rw8M3cUWjEaeZGSf0bjH5u4Y0sELsG572FQl6BRoRjMkaUU/ZlDHP/KzDMG38TkNQQkjEwQ1KY0PEquo6I2A+67zL7oYCt4HPMdbrogKbS+PQCcFWAihItfk/Pr1q4nOgGBJJleRVR+xM137wiv2i6sM9LMM1sNhls7igRu8UXmaYJ2sjrb6r2idPXU+NMuGzKqGuM67c//dp5/905PTd6Tj4flfzj42z++vf/355PTvZ5dv9dPp1eX5xS9vLz44lsOaFp4+7Ba3xtSedTiPHbQnelWGOWlXA3B1fv7rxeXZ3DmGcz5+OLm8Ob/68P7sw439Riv4Fg7dPWlcBDnA2kEr9+gBA8i8RDNn5xeH1muTYorIaR7e2z4RDXhYp8DMbu03P5fePf309kRUSeNnY12KUr2acqTtT68/Td5f34hlEMeLIHxwWrDoEoEHbksUg/+CqU0QMP1oz8S1cLXCHoNzcAXz/4F2rrFIe8715He8jWtt6Z2EEvGiX1lXRBH0qUi8BvH2WbMgJ/tNTcMtZLwci8MgX4G2OTx8eMRf/ZIOj2O60zWlb0qvcvV7NohOu5kCfRNZZyUKgah22gE5QuJF2AYAXMoU1FX01rQsR7Cd22tTbr8T9kFv1ccJBpaT307O5k0VfhlXxb3lVneSuUWVUcnwWXR1EdVGNqDH3wTSp1to1DK2dIbHnNe/3m6KAjba1G8nAjQbwSzDULj+Z/5tcZBpYtFv3O4GXcbxMYr2/RcxSI0+s2gnmlqgs19LtVAgPT/5iLrWdMqksQe8Hwr8wy9u6xYI1HjcB2KN9VZpqnQA23+LLsxwED/punyPFjChQwOa+RdxCooEfWJtLl2IBjapgvjENOR8Z7cBsQ9KXUBH7CZwXt/TBgc1h2fXIODcXkmCwyLtO7YCHTPI5nOlK8VpO7xvhrvTdM5gKEsANlrmSptCDWcWB4lrjtRGYVw3ydxO77pLjXXCtPrv6cI4SphYN6Lm1z7P3PTmoHCRPM8NL3B/zw7PH1i01V2jUx/tBp05Fmpgl7lTkPrxUf04PVfdwiAgzrqI9qOaEZ0EA0YD/HQaoFxjVef7rapmu7lGZKO4SP80t92NEMHdIaADE7UELNYH7uke0SpOY/ugg0SNPeCeeYOMcc1984b8Q0Std+tiH5ujUKKrHPQt5Q/yuRVL8BtNjzLfNqpCs56RNpbxLhkwE5GV2PAnz+gnEhAtM5o2a68eY4LjXVbF3FmCDwFSNOYV86XzlWwCPaGiRNfL958ggqKhVtsI2lJ2CVEbxRbs38hYxh1HH8OnZE+vJhmmFWbKqQh5bJciXlRc+P+oXdV1qABU6HqOgRHHZPOvBPzTuFOg2p+ms/b9w0UpteyWo8AVIiD2ZbmumDsjiP+o9QBdonwjdWtoEJa1AeLdeUfHkA/0wEOwkm4rgHo5iTC8aBOdAg237n30Y+yVSyjaHzctkT7AuVHy0Vtnr5GErrNWn6kFwczE59Ykyy7/HzBGLeR7q5stysZqAYDFgFPaacGjx//06qG7526ag5YCvKJu5l/UMFK47dhNNyJozNZ9GmCSiiouvS8qcxCDNW7R0PUQ3qZDC9ujPbe1z1tKGaEt6J9oE2qQdhbxKOrWvbXev6rsHFtkalTxfZ3PlKzi5CNWt+r5F9f+27PzX0Hw3nKHB7i8aiPbPtuSuh1j0GMbDnwPNZ7R5URUgs+sF3KqsTnfLBvXv4yktAk0szMUrm2t6+5HY4ImrRSH5eZMevq1dQt7ZsF4GNpkOp36J0RzH9t/pxPNFmaqNloT/bfOgZmuGjIoNWXdbh7l71JmIhDFGiyGwGwn2l6KYhkdIlhiPv3q6v0Et4L9m0bzdVBQazh2h4RBBZbd5E96ifAdPNbqqcJYnjzAwn2Om3Dun8JEJnMyRPtu52CLha2GvZ2sxqxOszU1NFWlTmp1K/yHu4r8Y+Pla89YM+fAJlitCDIXlNYCO8xmutfVamsFuR5aaKqpL8x8nV69f3/x8Wa8t8PgG1JfbJnIafW1CbCWQBSyWsmodUKdi8MW5wg9QZurdC7aIhZbWGSjy7Q8B1sZmYYOXfLEZUscBy+Llxs3i9MRtJPVcGhc3NbXACyFVluyWWg1MmMJzKq325Ow0uJjQkL3o7G/vc9NaH1tYhInkVousY+bP5UAh6EgICF4M4n+xhBpGxtxIvHP7vqedbupTaLOHDvq08i+HrVTm1Jj66sY3SGvP2jh5G+ab7+tJq2VHtWZyCXHegbreTjLfOJBRQ4rfGNbAJd2ZvT5ixeBVsUfLtOtZ34dE5j5WMFVuE4L1C3FYFhp6kRUvULDsxFW/1ROTuEHPkUT7upaD3aO4r3w4yDsEFRJlVaUGGs+EqKiVSQzcKJRlJsPhJAbzLdB4reTM2pYKBXlQsF8UDjiiXdBHj0CCSBAKYn5zRcZcEBB3ku4xY+MwBIlEuvL3JpVSiK05+wpMgyYZyblQCRnlYWGW61aCY2x6DQs2i2deEI3hMPkREILyS2Yt0se33oDS3K0KZgN5DeXzvWvJ5cz+prrqzn49rt2uei7u6fC0JRS079cfxL190LPZz0ZrIPuyVeXZ+L65OZmZp/cnDnzXi+fdMPu+BngdPacObT5Fq3msR0wWs0i6J33O8Z1CzKZRJ+Uje9blpECiiMuO5HHf4SrnKauwBOAZi3/f9RkBolVKDXGik/mS5/CMcxzNnxTG7KmgQPs4HxnoacfVjTsd+swwCb9QbnLgdNN55O2nr3UmmWI3KGUXc+XHbWMlM2k1kzK/GtNypl/4wJFDnZT4ZcLzfRbB6x3gW1aMPP4xX3XKaksjOmpC9tK6zMjqWSThpx3bOCEkzCh2Dv7dnpnfShDH8AM5zB7frljUQYddMqgGp+k/s4Gfo+pA6aFMr3CtJSROSYnf0fpBGfATredKvQdzqhFq/vyOYTaBRENEJ/MAQ76DGqpGJM6sLAQymlsakEAe0QBL24BUMiypIaIu1uHJ6F9G3rLS61MuzQkRoPcfJ/YS83rLyOxFj0e+iRy1weRwx0cdl29xmWn5I5HlhD/UBIf7WYsV2n7u80gbn1f2TsGHGqIs3z+AFF9MUeSN4K7g6bD8MoaghALBLr0G0zwm+H9zWy8QzGwC374UQUraV492QyJjfdMLGLBB7kl7qOeeXIXYWSsH8FhNITyFH4p5o6e/WjihG1764MEOg9kOC+04zrgt7KfNTENPy0hYvbROoc+lCWY8cMi/G7KervANm2/UF+0UB13JyCOclDM6ZpkG2bp3jXC1HOX+5TUjGvlW3GjCZ0saqa3jBdH7zs18XCUP6yDycNDM4g/iFr0g76M49V0SZqGBRNQuIg+eDTooXoA3Bo/hwILinVHLFFaookRcWu3lpbXb1jJt6ZpOepptO5nGEOLSNZ4TasU6pgWa7pRdydk5VAr0r07hAMXYZ8HL0JoZBzCIaOXfxbUmCZsOmGbNNG1B9GuOBjEtvuz20q928h01/uApdshVeeprMas0V5L8fItWy27nU1bDVV3zyLsgowMfp614q8qMLqzrblVPxzZJqIpT9RfrNbWBYNUdPRQYTfgD2VSG7hb7Vl3O47av9nA21Z+dtdhzfdcz90Qc7C9wzo59uevNBzCND7auFZV877BxhIecbGvuXiu/96VbbZssK7V2V5O762zM22dyeCBPygGWUTfDpysxRYCxdaGO2e9ZGNdZtm/b3vS7m2JDVkQ5j1Z6vMPOPxIyV0L2rn9XWcyv2EDoPWJIKwsc9DuZuzN2LAX88b8eZb51ui1U4g62B1VvrCKOsy7/4sKqk5w7/L3uwGQrbH10lYOqx2e8SNFZ5aBN+sGzfi3YhqzHbsS/5ZWISTXCMb2UNCx4i/iA7CA4i896spiLZsT1vzs0Uiz2PuDVegXXUd/9w2wmIw8tbVgPGZaXLyTfEW+9jW9cSPJGUjMm/t+lIYQ31sr8Z8S8AO9xG3ie+SQ+TXVFODIAAR9vitHoP85heL2+G7vzvU/olDvvXc65xQsOExa1Ir5d51Evbt6MbkzzS2me1daX1zZi/FfIJo7V2tVdj43M1mydfBZras1fi1iPNgdJ+hUFfXSEFHAIKdYx4VL7V9ZZ2EGFmoQaya4+XSNEeFNqxVG70t/4c6m5oqxDfYk8gF2T0ar65jmIHvYhRMaBFJ1/yUK2YoJzAlF/QUVf0SkIZLs7RgWwfeG2E7XMMM96pJRDdHY3r4BytSO+GU7Z8mDTQO3qSbRsH4YdTJ6zg1p1FktkbfGM8bY9z8E9yXgawMBiqzCHkDWsRQ8+NwW6WMsTZJ88D/PvFgJMEsAAA=="
  }
}
# END EMBEDDED CONTEXT HELPERS
TOOLS = Path(tempfile.mkdtemp(prefix="ace_context_tools_"))
for name, payload in EMBEDDED_HELPERS.items():
    data = gzip.decompress(base64.b64decode(payload["gzip_base64"], validate=True))
    assert hashlib.sha256(data).hexdigest() == payload["sha256"], f"Helper checksum mismatch: {name}"
    (TOOLS / name).write_bytes(data)
# Explicitly load this notebook's helper, even if an earlier notebook imported one.
spec = importlib.util.spec_from_file_location("electric_full_song", TOOLS / "electric_full_song.py")
embedded_single = importlib.util.module_from_spec(spec)
spec.loader.exec_module(embedded_single)
sys.modules["electric_full_song"] = embedded_single
spec = importlib.util.spec_from_file_location("ace_context_embedded", TOOLS / "ace_context_chunks.py")
context = importlib.util.module_from_spec(spec)
spec.loader.exec_module(context)
ROOT = context.single.discover_root(Path.cwd(), globals().get("ROOT"))
if ROOT is None:
    raise RuntimeError("Checkout bridge belum ada di VM ini. Setup notebook 10 dahulu; notebook ini belum menyediakan model/GPU")

print("Upload satu file piano asli saja; tidak perlu ZIP tools")
uploaded = files.upload()
audios = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
assert len(audios) == 1, "Pilih satu file piano asli"
SOURCE = TOOLS / ("piano" + Path(audios[0]).suffix.lower())
SOURCE.write_bytes(uploaded[audios[0]])
OUT = ROOT / "runs" / ("ace_context_" + uuid.uuid4().hex[:10])
plan = context.prepare(SOURCE, OUT, context=90, margin=6, fade=.58)
PLAN_PATH = OUT / "plan.json"
print("ROOT:", ROOT, "| PLAN:", PLAN_PATH)
for chunk in plan["chunks"]:
    print(chunk["index"], "input model:", chunk["model_duration"], "detik", "core:", round((chunk["core_end"]-chunk["core_start"])/32000, 3))
print("Tanpa SongMap terverifikasi: batas ini bukan downbeat atau bar yang diketahui")


## Satu probe durasi

Opsional: `90` sudah dipilih, sehingga tidak perlu diulang. `60` tersedia sebagai kontrol; `120` hanya untuk eksperimen eksplisit meskipun sudah dilaporkan berisik. Caption, seed, FP32, CFG dan langkah tetap sama. Dengarkan bagian waktu yang sama, jangan hanya melihat status generated. Jangan menjalankan beberapa probe sekaligus di T4 yang sama.


In [ ]:
from IPython.display import Audio, display

DURATION = 90  # 60 / 90 / 120
EXECUTE_PROBE = False  # True hanya pada runtime CUDA/bobot yang siap
PROBE_OUT = OUT / (f"probe_{DURATION}_" + uuid.uuid4().hex[:8])
probe_report = context.single.generate(ROOT, context.generation_source(OUT, plan), PROBE_OUT,
    start=0, duration=DURATION, execute=EXECUTE_PROBE)
print("Status:", probe_report["status"], "|", PROBE_OUT)
if EXECUTE_PROBE:
    display(Audio(filename=str(PROBE_OUT / "mix_preview.mp3"), normalize=False))


In [ ]:
# Jalankan setelah probe selesai, atau setelah gagal untuk report yang tersedia.
feedback = PROBE_OUT / "full_song_feedback.zip"
if feedback.is_file():
    files.download(str(feedback))
else:
    print("Belum ada feedback inferensi. Probe dry-run tidak membuat hasil model.")


## Chunk seluruh lagu, setelah durasi konteks dipilih

Plan awal membatasi **seluruh konteks model menjadi 90 detik termasuk margin**. Jika memilih cap lain, buat plan baru dengan `context.prepare(SOURCE, NEW_OUT, context=60, margin=6, fade=.58)` dan perbarui `PLAN_PATH`. Jangan mengganti plan yang telah memiliki hasil.

Generasi sequential dan memuat model lagi per chunk. Margin/reload menambah waktu. Chunk selesai dipakai kembali setelah verifikasi; proses diffusion yang gagal diulang sebagai attempt baru. Seed sama tidak menjamin timbre/register/groove sama.


In [ ]:
EXECUTE_CHUNKS = False
if EXECUTE_CHUNKS:
    progress = context.run_chunks(ROOT, PLAN_PATH)
    report = context.stitch_run(PLAN_PATH)
    print("Sambungan perlu didengar:", [s["index"] for s in report["seams"] if s["manual_review"]])
    display(Audio(filename=str(PLAN_PATH.parent / "stitched/mix_full.mp3"), normalize=False))
else:
    print("Tidak menjalankan inferensi atau stitching; plan tersimpan di", PLAN_PATH)


In [ ]:
# Paket kecil: mix/bass MP3, report seam dan cuplikan seam; tanpa bobot model.
stitched = PLAN_PATH.parent / "stitched"
if (stitched / "seams.json").is_file():
    bundle = PLAN_PATH.parent / "chunked_feedback.zip"
    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(PLAN_PATH, "plan.json")
        for path in sorted(stitched.iterdir()):
            if path.suffix in {".json", ".mp3"} or path.name.startswith("seam_"):
                archive.write(path, "stitched/" + path.name)
    files.download(str(bundle))
else:
    print("Belum ada hasil stitching lengkap.")


Equal-power crossfade **tetap dapat membatalkan fase bass**. Helper mencari kandidat berdasarkan energi/korelasi, membuang tepi model, dan melaporkan perubahan level/pitch; ia tidak menjamin sambungan musikal. Birama/downbeat tidak ditebak dari BPM. Repaint dan conditioning BPM/key belum diterapkan. Lihat `ace-context-chunks.md` di tool ZIP untuk rincian.
